# 🌴 Date-Vision: Date Fruit Detection & Sorting with YOLOv8 (Google Colab)

An end-to-end pipeline that **detects and classifies 9 Saudi date varieties** with a fine-tuned YOLOv8 model and exports it to ONNX for real-time deployment.

**Before you start**
1. Go to **Runtime → Change runtime type → T4 GPU**.
2. Add your Kaggle credentials as **Colab Secrets** `KAGGLE_USERNAME` and `KAGGLE_KEY` (🔑 icon in the left sidebar; see Section 4).
3. Run the cells **from top to bottom** (Runtime → Run all).
4. All training outputs (runs, weights, metrics, plots, ONNX model) are saved to **Google Drive** under `MyDrive/Date_Vision_Project`.

> 💡 The Kaggle archive contains class folders (`Ajwa/`, `Sokari/`, …) with **no box labels**. Section 7 auto-generates YOLO boxes and writes `date_fruits.yaml` inside the extracted folder. If you point `KAGGLE_DATASET` at a dataset that already ships a YOLO `data.yaml`, that file is detected and used directly.

> ⚠️ Files in `/content` are **erased when the runtime resets**, but every output on Drive is kept. After a disconnect, re-run Sections 1–7 (the download takes about 1–2 min), then set `RESUME = True` in Section 8 to continue training from the last checkpoint on Drive.

| # | Section |
|---|---|
| 1 | GPU check |
| 2 | Mount Google Drive & project configuration |
| 3 | Install dependencies |
| 4 | Authenticate Kaggle (Colab Secrets) |
| 5 | Download & unzip the dataset |
| 6 | Auto-labelling & helper functions |
| 7 | Prepare the dataset & locate the `.yaml` |
| 8 | Train YOLOv8 |
| 9 | Evaluate (mAP50, mAP50-95, confusion matrix) |
| 10 | Inference & error analysis (TP + overlapping-fruit failures) |
| 11 | Export to ONNX + benchmark |
| 12 | Try your own image |

## 1 · GPU check

In [ ]:
# If this fails: Runtime → Change runtime type → Hardware accelerator: T4 GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2 · Mount Google Drive & project configuration

**Storage strategy**
* **Google Drive** (`PROJECT_ROOT`) holds every output you keep: weights, training runs, evaluation plots, showcase images, the ONNX export and QA files.
* **Colab local disk** (`/content`) holds the downloaded Kaggle zip and the extracted images. Training reads from local disk because reading thousands of small files from Drive is roughly 10× slower.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# >>>>>>>>>>>>>>>>  The ONLY path you normally need to change  <<<<<<<<<<<<<<<<
PROJECT_ROOT = Path('/content/drive/MyDrive/Date_Vision_Project')

# ── Persistent outputs: Google Drive ─────────────────────────────────────────
DRIVE_DATA      = PROJECT_ROOT / 'data'
QA_DIR          = DRIVE_DATA / 'qa'                     # auto-label QA grid + review CSV
WEIGHTS_DIR     = PROJECT_ROOT / 'weights'              # pretrained yolov8*.pt
RUNS_DIR        = PROJECT_ROOT / 'runs'
TRAIN_PROJECT   = RUNS_DIR / 'train'                    # -> model.train(project=..., name=RUN_NAME)
EVAL_PROJECT    = RUNS_DIR / 'eval'
PREDICT_PROJECT = RUNS_DIR / 'predict'
EXPORT_DIR      = PROJECT_ROOT / 'exports'              # .onnx + export_info.json
SHOWCASE_DIR    = PROJECT_ROOT / 'showcase'             # curated TP / failure images

# ── Dataset: Colab local disk (fast I/O, wiped on runtime reset) ─────────────
# Kaggle dataset: https://www.kaggle.com/datasets/wadhasnalhamdan/date-fruit-image-dataset-in-controlled-environment
KAGGLE_DATASET  = 'wadhasnalhamdan/date-fruit-image-dataset-in-controlled-environment'
ZIP_PATH        = Path('/content') / f"{KAGGLE_DATASET.split('/')[1]}.zip"   # where `kaggle datasets download` saves it
EXTRACT_DIR     = Path('/content/dataset')              # the zip is extracted here
AUTO_YOLO_DIR   = EXTRACT_DIR / 'yolo_autolabelled'     # generated only if the dataset has no .yaml
STRESS_DIR      = Path('/content/stress_overlap')       # synthetic overlapping-fruit test set
DATA_YAML       = None                                  # set in Section 7 (the .yaml used for training)

for d in (QA_DIR, WEIGHTS_DIR, TRAIN_PROJECT, EVAL_PROJECT, PREDICT_PROJECT, EXPORT_DIR, SHOWCASE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Keep Ultralytics settings/fonts on Drive (must be set BEFORE `import ultralytics`)
os.environ['YOLO_CONFIG_DIR'] = str(PROJECT_ROOT / 'cache' / 'ultralytics')

# ── Experiment ───────────────────────────────────────────────────────────────
SEED        = 42
SPLIT_RATIOS = (0.70, 0.15, 0.15)       # train / val / test, stratified per class
MAX_SIDE     = 1280                     # downscale 5184 px DSLR frames
IMGSZ        = 640
BASE_MODEL   = 'yolov8s.pt'             # n / s / m / l / x
RUN_NAME     = 'yolov8s_dates'
BEST_PT      = TRAIN_PROJECT / RUN_NAME / 'weights' / 'best.pt'
LAST_PT      = TRAIN_PROJECT / RUN_NAME / 'weights' / 'last.pt'

# ── Training hyper-parameters ────────────────────────────────────────────────
TRAIN_HYP = dict(
    epochs=100, imgsz=IMGSZ, batch=16,     # T4 (16 GB): batch 16-32 for yolov8s
    patience=20, optimizer='AdamW', lr0=1e-3, lrf=0.01, cos_lr=True,
    momentum=0.937, weight_decay=5e-4, warmup_epochs=3, close_mosaic=10,
    seed=SEED, deterministic=True, amp=True, plots=True,
    workers=2,                             # Colab VMs have 2 vCPUs
    cache='ram',                           # ~1.5 GB of RAM -> much faster epochs
    save_period=5,                         # extra epochN.pt on Drive every 5 epochs

    # ── AUGMENTATION: simulate a factory / conveyor environment ──────────────
    hsv_h=0.010,        # hue kept SMALL: colour separates varieties (e.g. Rutab vs Sokari)
    hsv_s=0.50,         # white-balance drift between cameras
    hsv_v=0.40,         # BRIGHTNESS: flickering / ageing LED lighting
    degrees=30.0,       # ROTATION: fruit lands at arbitrary angles on the belt
    translate=0.10, scale=0.50, shear=2.0, perspective=0.0005,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0,         # 4 images stitched -> several fruits per frame
    mixup=0.10,         # partial overlap between fruits
)

CONF_THRES, NMS_IOU, MATCH_IOU = 0.25, 0.50, 0.50
print('✓ Project root:', PROJECT_ROOT)

## 3 · Install dependencies
Colab already ships **PyTorch (CUDA), torchvision, OpenCV, NumPy, PyYAML and Matplotlib**. Do **not** reinstall torch, because that can replace the CUDA build.

In [ ]:
!pip install -q "ultralytics>=8.3.0,<8.4" kaggle onnx onnxslim onnxruntime-gpu

import torch, cv2, ultralytics
ultralytics.checks()
print(f'torch {torch.__version__} | CUDA available: {torch.cuda.is_available()} | OpenCV {cv2.__version__}')
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

## 4 · Authenticate Kaggle (Colab Secrets)

**One-time setup:**
1. On [kaggle.com](https://www.kaggle.com) go to **Settings → API → Create New Token**. This downloads `kaggle.json`, which contains your `username` and `key`.
2. In Colab, click the 🔑 **Secrets** icon in the left sidebar and add two secrets:
   * `KAGGLE_USERNAME`: the `username` value from `kaggle.json`
   * `KAGGLE_KEY`: the `key` value from `kaggle.json`
3. Turn on **Notebook access** for both secrets.

⚠️ Never paste your key directly into a cell. Secrets keep it out of the notebook file and out of GitHub.

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
    os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
except Exception as e:   # SecretNotFoundError / NotebookAccessError
    raise RuntimeError(
        'Kaggle secrets not available. Add KAGGLE_USERNAME and KAGGLE_KEY in the 🔑 Secrets '
        'sidebar and enable "Notebook access" for both, then re-run this cell.') from e

print(f"✓ Kaggle credentials loaded for user: {os.environ['KAGGLE_USERNAME']}")

## 5 · Download & unzip the dataset

Downloads the dataset (about 3.4 GB, usually 1–2 min on Colab) to `/content` with the Kaggle CLI and extracts it to `/content/dataset`.
Both steps are skipped if the files are already present in this runtime. Set `FORCE_DOWNLOAD = True` to fetch it again.

In [ ]:
import shutil, zipfile

FORCE_DOWNLOAD = False

if FORCE_DOWNLOAD or not ZIP_PATH.exists():
    !kaggle datasets download -d {KAGGLE_DATASET} -p /content --force
if not ZIP_PATH.exists() or not zipfile.is_zipfile(ZIP_PATH):
    raise FileNotFoundError(
        f'{ZIP_PATH} was not downloaded. Check the Kaggle output above:\n'
        '  401 Unauthorized -> wrong KAGGLE_USERNAME / KAGGLE_KEY in Colab Secrets\n'
        '  403 Forbidden    -> open the dataset page on kaggle.com while logged in and accept its terms\n'
        '  404 Not Found    -> check KAGGLE_DATASET in Section 2')
print(f'✓ {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e9:.2f} GB)')

if FORCE_DOWNLOAD or not (EXTRACT_DIR.exists() and any(EXTRACT_DIR.iterdir())):
    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
    EXTRACT_DIR.mkdir(parents=True)
    print(f'Extracting -> {EXTRACT_DIR} ...')
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_DIR, members=[m for m in zf.namelist() if '__MACOSX' not in m])

n_imgs = sum(1 for p in EXTRACT_DIR.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'})
print(f'✓ {n_imgs} images under {EXTRACT_DIR}')
!ls "{EXTRACT_DIR}"

## 6 · Auto-labelling & helper functions

**Why auto-labelling:** the original Kaggle dataset is *classification-only*. It has 9 variety folders, **one date per image** on a white background, and **no bounding boxes**. Because the background is controlled, classical OpenCV segmentation produces accurate boxes automatically:

`Otsu (inverted grey) ∪ saturation threshold → morphology → largest centred contour → padded box`

Images that cannot be segmented reliably are excluded and listed in `qa/needs_review.csv`. *(This step is skipped if your zip already contains YOLO labels and a `.yaml`.)*

*Optional quality grading:* if `data/annotations/quality_labels.csv` exists on Drive (columns `image,quality`, where `image` is the path inside the zip, e.g. `Ajwa/Ajwa Date (1).JPG`), classes become `<Variety>_<quality>`, for example `Ajwa_damaged`.

In [ ]:
import csv, json, random, shutil
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

import cv2
import numpy as np
import yaml
import matplotlib.pyplot as plt

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
SPLITS = ('train', 'val', 'test')
QUALITY_CSV = DRIVE_DATA / 'annotations' / 'quality_labels.csv'


def segment_fruit(img_bgr, work_side=512):
    """Return (full-res mask, xyxy box) of the dominant fruit; box is None if implausible."""
    h, w = img_bgr.shape[:2]
    s = min(1.0, work_side / max(h, w))
    small = cv2.resize(img_bgr, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else img_bgr
    sh, sw = small.shape[:2]

    gray = cv2.GaussianBlur(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, dark = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    _, saturated = cv2.threshold(cv2.cvtColor(small, cv2.COLOR_BGR2HSV)[..., 1], 60, 255, cv2.THRESH_BINARY)
    mask = cv2.bitwise_or(dark, saturated)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)    # remove specks
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=3)   # fill flash highlights

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return np.zeros((h, w), np.uint8), None

    def score(c):
        x, y, cw, ch = cv2.boundingRect(c)
        touches = x <= 1 or y <= 1 or x + cw >= sw - 1 or y + ch >= sh - 1
        return cv2.contourArea(c) * (0.2 if touches else 1.0)          # ignore vignette corners

    best = max(contours, key=score)
    clean = np.zeros((sh, sw), np.uint8)
    cv2.drawContours(clean, [best], -1, 255, thickness=cv2.FILLED)
    full_mask = cv2.resize(clean, (w, h), interpolation=cv2.INTER_NEAREST)

    x, y, cw, ch = cv2.boundingRect(best)
    if cv2.contourArea(best) / (sh * sw) < 0.005 or (cw * ch) / (sh * sw) > 0.95:
        return full_mask, None
    pad = 0.02
    box = (max(0, int((x - pad * sw) / s)), max(0, int((y - pad * sh) / s)),
           min(w, int((x + cw + pad * sw) / s)), min(h, int((y + ch + pad * sh) / s)))
    return full_mask, box


def resize_max_side(img, max_side=MAX_SIDE):
    h, w = img.shape[:2]
    s = min(1.0, max_side / max(h, w))
    return cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA) if s < 1 else img


def to_yolo_line(cls_id, box, w, h):
    x1, y1, x2, y2 = box
    return f'{cls_id} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}'


def label_path_for(img_path):
    """YOLO convention: replace the LAST '/images/' with '/labels/' and the extension with .txt.
    Works for  images/train/x.jpg -> labels/train/x.txt  and  train/images/x.jpg -> train/labels/x.txt"""
    s = Path(img_path).as_posix()
    head, sep, tail = s.rpartition('/images/')
    return Path(head + '/labels/' + tail if sep else s).with_suffix('.txt')


def show_images(paths, titles=None, cols=2, size=7):
    paths = [Path(p) for p in paths]
    if not paths:
        print('(no images to show)')
        return
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, p, t in zip(axes.flat, paths, titles or [p.name for p in paths]):
        ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        ax.set_title(t, fontsize=11)
    plt.tight_layout()
    plt.show()


print('✓ helpers defined')

## 7 · Prepare the dataset & locate the `.yaml`

This cell decides automatically which **`.yaml`** the training cell will use:

| What's inside the downloaded dataset | What happens | `DATA_YAML` used for training |
|---|---|---|
| A YOLO dataset with a `data.yaml` (e.g. a Roboflow export) | The yaml is found and its split paths are made absolute for Colab. A `test` split falls back to `val` if missing. | `/content/dataset/.../data_colab.yaml` (next to the original) |
| Class folders only (the original Kaggle archive) | Auto-labels every image and makes a stratified 70/15/15 split | `/content/dataset/yolo_autolabelled/date_fruits.yaml` |

Set `FORCE_REBUILD = True` to regenerate auto-labels.

In [ ]:
FORCE_REBUILD = False


# ─────────────────────────── Case A: zip already in YOLO format ──────────────
def find_yolo_yaml():
    """Return the first .yaml in the extracted folder that defines `names` and `train`."""
    for y in sorted(EXTRACT_DIR.rglob('*.y*ml')):
        if AUTO_YOLO_DIR in y.parents or y.name == 'data_colab.yaml':
            continue
        try:
            cfg = yaml.safe_load(y.read_text())
        except Exception:
            continue
        if isinstance(cfg, dict) and 'names' in cfg and 'train' in cfg:
            return y, cfg
    return None, None


def resolve_split(value, yaml_dir, base):
    """Resolve split paths written as `../train/images`, `train/images`, `images/train`, etc."""
    if not value:
        return None
    value = value[0] if isinstance(value, list) else value
    candidates = [Path(value)] if Path(value).is_absolute() else [
        (base / value), (yaml_dir / value), (yaml_dir / value.lstrip('./').replace('../', '', 1)),
        (yaml_dir / value.replace('../', '')),
    ]
    for c in candidates:
        c = c.resolve()
        if c.exists():
            return c
    return None


def prepare_from_yaml(src_yaml, cfg):
    yaml_dir = src_yaml.parent
    base = (yaml_dir / cfg['path']).resolve() if cfg.get('path') and not Path(cfg['path']).is_absolute() \
        else Path(cfg.get('path') or yaml_dir)
    splits = {sp: resolve_split(cfg.get(key), yaml_dir, base)
              for sp, key in (('train', 'train'), ('val', 'val'), ('test', 'test'))}
    if splits['val'] is None and 'valid' in cfg:
        splits['val'] = resolve_split(cfg['valid'], yaml_dir, base)
    if splits['train'] is None or splits['val'] is None:
        raise FileNotFoundError(f'Could not resolve train/val folders from {src_yaml}: {cfg}')
    if splits['test'] is None:
        print('⚠️  No test split in the yaml — evaluation will use the val split.')
        splits['test'] = splits['val']

    names = cfg['names']
    names = {i: n for i, n in enumerate(names)} if isinstance(names, list) else {int(k): v for k, v in names.items()}
    out = yaml_dir / 'data_colab.yaml'
    out.write_text(yaml.safe_dump({'path': str(yaml_dir), **{k: str(v) for k, v in splits.items()},
                                   'names': names}, sort_keys=False))
    print(f'✓ YOLO-format dataset detected: {src_yaml}')
    return out


# ─────────────────────────── Case B: class folders → auto-label ──────────────
@dataclass
class Sample:
    src: Path
    rel: str
    cls_name: str
    split: str = ''
    dst_stem: str = ''


def discover_samples():
    quality = None
    if QUALITY_CSV.exists():
        with QUALITY_CSV.open(newline='', encoding='utf-8') as f:
            quality = {r['image'].replace('\\', '/'): r['quality'].strip().lower() for r in csv.DictReader(f)}
        print(f'Quality labels found ({len(quality)}) -> classes = <Variety>_<quality>')
    samples = []
    for p in sorted(EXTRACT_DIR.rglob('*')):
        if (not p.is_file() or p.suffix.lower() not in IMG_EXTS or p.name.startswith('._')
                or AUTO_YOLO_DIR in p.parents):
            continue
        rel = p.relative_to(EXTRACT_DIR).as_posix()
        cls = '_'.join(p.parent.name.split())             # "Nabtat Ali" -> "Nabtat_Ali"
        if quality is not None:
            key = next((k for k in quality if rel.endswith(k)), None)
            if key is None:
                continue
            cls = f'{cls}_{quality[key]}'
        samples.append(Sample(p, rel, cls))
    if not samples:
        raise FileNotFoundError(f'No images under {EXTRACT_DIR} — run Section 5.')
    return samples


def stratified_split(samples):
    rng = random.Random(SEED)
    by_cls = defaultdict(list)
    for s in samples:
        by_cls[s.cls_name].append(s)
    for cls, items in by_cls.items():
        rng.shuffle(items)
        n_tr = round(len(items) * SPLIT_RATIOS[0])
        n_va = round(len(items) * SPLIT_RATIOS[1])
        for i, s in enumerate(items):
            s.split = 'train' if i < n_tr else 'val' if i < n_tr + n_va else 'test'
            s.dst_stem = f'{cls}_{i:04d}'


def process(sample, cls_id):
    img = cv2.imread(str(sample.src), cv2.IMREAD_COLOR)
    if img is None:
        return dict(ok=False, sample=sample, reason='unreadable')
    img = resize_max_side(img)
    _, box = segment_fruit(img)
    if box is None:
        return dict(ok=False, sample=sample, reason='segmentation_failed')
    h, w = img.shape[:2]
    out = AUTO_YOLO_DIR / 'images' / sample.split / f'{sample.dst_stem}.jpg'
    cv2.imwrite(str(out), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    (AUTO_YOLO_DIR / 'labels' / sample.split / f'{sample.dst_stem}.txt').write_text(to_yolo_line(cls_id, box, w, h) + '\n')
    return dict(ok=True, sample=sample, box=box, img=out)


def build_autolabelled_dataset():
    out_yaml = AUTO_YOLO_DIR / 'date_fruits.yaml'
    if out_yaml.exists() and not FORCE_REBUILD:
        print(f'✓ Auto-labelled dataset already built: {out_yaml}')
        return out_yaml
    for sub in ('images', 'labels'):
        shutil.rmtree(AUTO_YOLO_DIR / sub, ignore_errors=True)
        for sp in SPLITS:
            (AUTO_YOLO_DIR / sub / sp).mkdir(parents=True, exist_ok=True)

    samples = discover_samples()
    class_names = sorted({s.cls_name for s in samples})
    cls_to_id = {c: i for i, c in enumerate(class_names)}
    stratified_split(samples)
    print(f'No .yaml in the dataset -> auto-labelling {len(samples)} images | {len(class_names)} classes: {class_names}')

    with ThreadPoolExecutor(max_workers=8) as pool:      # cv2 releases the GIL
        results = list(pool.map(lambda s: process(s, cls_to_id[s.cls_name]), samples))
    ok = [r for r in results if r['ok']]
    bad = [r for r in results if not r['ok']]

    with (QA_DIR / 'manifest.csv').open('w', newline='') as f:
        wr = csv.writer(f)
        wr.writerow(['source', 'class', 'split', 'image', 'x1', 'y1', 'x2', 'y2'])
        for r in ok:
            wr.writerow([r['sample'].rel, r['sample'].cls_name, r['sample'].split, r['img'].name, *r['box']])
    with (QA_DIR / 'needs_review.csv').open('w', newline='') as f:
        wr = csv.writer(f)
        wr.writerow(['source', 'class', 'reason'])
        for r in bad:
            wr.writerow([r['sample'].rel, r['sample'].cls_name, r['reason']])

    counts = Counter((r['sample'].cls_name, r['sample'].split) for r in ok)
    print(f"\n{'class':<20}{'train':>7}{'val':>6}{'test':>6}")
    for c in class_names:
        print(f"{c:<20}{counts[(c, 'train')]:>7}{counts[(c, 'val')]:>6}{counts[(c, 'test')]:>6}")
    print(f'\nLabelled {len(ok)}/{len(results)} | excluded {len(bad)} -> {QA_DIR / "needs_review.csv"}')

    out_yaml.write_text(yaml.safe_dump({
        'path': str(AUTO_YOLO_DIR), 'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
        'names': {i: c for i, c in enumerate(class_names)}}, sort_keys=False))
    return out_yaml


def split_dir(split):
    """Absolute image folder of a split, as declared in DATA_YAML."""
    cfg = yaml.safe_load(DATA_YAML.read_text())
    p = Path(cfg[split])
    return p if p.is_absolute() else Path(cfg['path']) / p


def save_qa_grid(n=16, tile=320):
    names = yaml.safe_load(DATA_YAML.read_text())['names']
    imgs = sorted(p for p in split_dir('train').glob('*') if p.suffix.lower() in IMG_EXTS)
    tiles = []
    for p in random.Random(SEED).sample(imgs, min(n, len(imgs))):
        img = cv2.imread(str(p))
        h, w = img.shape[:2]
        lp = label_path_for(p)
        for line in (lp.read_text().splitlines() if lp.exists() else []):
            c, cx, cy, bw, bh = map(float, line.split()[:5])
            cv2.rectangle(img, (int((cx - bw / 2) * w), int((cy - bh / 2) * h)),
                          (int((cx + bw / 2) * w), int((cy + bh / 2) * h)), (0, 200, 0), max(2, w // 200))
            cv2.putText(img, str(names[int(c)]), (int((cx - bw / 2) * w), max(30, int((cy - bh / 2) * h) - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, max(0.8, w / 600), (0, 0, 255), max(2, w // 300))
        tiles.append(cv2.resize(img, (tile, tile)))
    while len(tiles) % 4:
        tiles.append(np.full((tile, tile, 3), 255, np.uint8))
    grid = np.vstack([np.hstack(tiles[i:i + 4]) for i in range(0, len(tiles), 4)])
    out = QA_DIR / 'label_qa_grid.jpg'
    cv2.imwrite(str(out), grid)
    return out


src_yaml, src_cfg = find_yolo_yaml()
DATA_YAML = prepare_from_yaml(src_yaml, src_cfg) if src_yaml else build_autolabelled_dataset()
shutil.copy2(DATA_YAML, QA_DIR / DATA_YAML.name)            # keep a copy of the yaml on Drive

CLASS_NAMES = yaml.safe_load(DATA_YAML.read_text())['names']
print(f'\n✓ DATA_YAML = {DATA_YAML}')
print('Classes:', CLASS_NAMES)
print({sp: sum(1 for p in split_dir(sp).glob('*') if p.suffix.lower() in IMG_EXTS) for sp in SPLITS})
show_images([save_qa_grid()], ['Label QA grid (check every box hugs its fruit)'], cols=1, size=12)

## 8 · Train YOLOv8

* `data=DATA_YAML` is the `.yaml` that Section 7 located or generated **inside `/content/dataset`**.
* Pretrained weights are downloaded to `Drive/weights/`.
* `project=TRAIN_PROJECT` and `name=RUN_NAME` mean checkpoints are written straight to Drive (`runs/train/yolov8s_dates/weights/{best,last}.pt`). They survive a disconnect.
* **Resume after a disconnect:** re-run Sections 1–7 (this re-downloads the dataset), set `RESUME = True`, and run this cell.

⏱️ About 100 epochs × 1,160 images on a T4 takes roughly 30–45 min (early stopping often ends it sooner). For a quick smoke test, set `TRAIN_HYP['epochs'] = 5`.

In [ ]:
from ultralytics import YOLO, settings

RESUME = False

settings.update({'datasets_dir': str(EXTRACT_DIR), 'weights_dir': str(WEIGHTS_DIR), 'runs_dir': str(RUNS_DIR)})

assert DATA_YAML is not None and Path(DATA_YAML).exists(), 'Run Section 7 first — DATA_YAML is not set.'
print('Training on:', DATA_YAML)
print(Path(DATA_YAML).read_text())

os.chdir(WEIGHTS_DIR)        # AMP self-check downloads a tiny model into the CWD -> keep it on Drive
try:
    if RESUME and LAST_PT.exists():
        print(f'Resuming from {LAST_PT}')
        YOLO(str(LAST_PT)).train(resume=True)
    else:
        model = YOLO(str(WEIGHTS_DIR / BASE_MODEL))
        model.train(
            data=str(DATA_YAML),
            project=str(TRAIN_PROJECT),     # Google Drive
            name=RUN_NAME,
            exist_ok=True,                  # stable path -> runs/train/yolov8s_dates
            device=DEVICE,
            **TRAIN_HYP,
        )
finally:
    os.chdir('/content')

print('✓ Best checkpoint:', BEST_PT)
run_dir = TRAIN_PROJECT / RUN_NAME
show_images([run_dir / 'results.png'], ['Training curves'], cols=1, size=14)

## 9 · Evaluate: mAP50, mAP50-95 & confusion matrix
The model is evaluated on the **held-out test split**, which was never seen during training or model selection. Results are saved to `Drive/runs/eval/yolov8s_dates_test/`.

In [ ]:
model = YOLO(str(BEST_PT))
metrics = model.val(
    data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=16,
    conf=0.001, iou=0.6,                      # standard COCO-style mAP protocol
    plots=True, device=DEVICE,
    project=str(EVAL_PROJECT), name=f'{RUN_NAME}_test', exist_ok=True,
)
eval_dir = Path(metrics.save_dir)
names = model.names

summary = {
    'mAP50': float(metrics.box.map50), 'mAP50-95': float(metrics.box.map),
    'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
    'speed_ms_per_img': metrics.speed, 'per_class': {},
}
for i, c in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(i)
    summary['per_class'][names[int(c)]] = dict(precision=float(p), recall=float(r), AP50=float(ap50), **{'AP50-95': float(ap)})

cm = np.asarray(metrics.confusion_matrix.matrix, dtype=int)   # [predicted, true], last = background
labels = [names[i] for i in range(len(names))] + ['background']
with (eval_dir / 'confusion_matrix.csv').open('w', newline='') as f:
    wr = csv.writer(f)
    wr.writerow(['pred \\ true', *labels])
    for lab, row in zip(labels, cm):
        wr.writerow([lab, *row.tolist()])
core = cm[:len(names), :len(names)]
summary['variety_accuracy_on_detected'] = float(np.trace(core) / max(core.sum(), 1))
(eval_dir / 'metrics.json').write_text(json.dumps(summary, indent=2))

print(f"mAP50    : {summary['mAP50']:.4f}\nmAP50-95 : {summary['mAP50-95']:.4f}")
print(f"Precision: {summary['precision']:.4f} | Recall: {summary['recall']:.4f}")
print(f"Variety accuracy (detected fruits): {summary['variety_accuracy_on_detected']:.4f}\n")
print(f"{'class':<20}{'P':>7}{'R':>7}{'AP50':>7}{'AP50-95':>9}")
for c, v in summary['per_class'].items():
    print(f"{c:<20}{v['precision']:>7.3f}{v['recall']:>7.3f}{v['AP50']:>7.3f}{v['AP50-95']:>9.3f}")
print('\nSaved to', eval_dir)

show_images([eval_dir / 'confusion_matrix_normalized.png'], ['Normalized confusion matrix (test)'], cols=1, size=11)

## 10 · Inference & error analysis

Each prediction is matched to the ground truth (IoU ≥ 0.5, one-to-one, class must match), and every image is placed into a bucket:
`true_positive` · `misclassified` · `missed` · `false_positive`. The annotated images are saved to `Drive/showcase/<set>/<bucket>/`.

**Overlapping-fruit stress set.** The Kaggle dataset has only **one** date per image, so occlusion never happens naturally. We composite 3–6 masked *test-split* fruits with 20–55% overlap onto a white canvas, keeping exact ground-truth boxes. This reproduces the typical sorting-line failure modes: merged boxes, missed occluded fruit, and NMS suppression.

🟩 Green = ground truth · coloured = prediction

In [ ]:
def list_images(split):
    return sorted(p for p in split_dir(split).glob('*') if p.suffix.lower() in IMG_EXTS)


def load_gt(img_path, w, h):
    gts, lp = [], label_path_for(img_path)
    if lp.exists():
        for line in lp.read_text().splitlines():
            if not line.strip():
                continue
            c, cx, cy, bw, bh = map(float, line.split()[:5])
            gts.append((int(c), np.array([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])))
    return gts


def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    tl = np.maximum(a[:, None, :2], b[None, :, :2])
    br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.prod(np.clip(br - tl, 0, None), axis=2)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def classify_outcome(pred_boxes, pred_cls, gts, iou_thr=MATCH_IOU):
    gt_boxes = np.array([g[1] for g in gts]).reshape(-1, 4)
    gt_cls = np.array([g[0] for g in gts], dtype=int)
    ious = iou_matrix(pred_boxes, gt_boxes)
    tp = mis = 0
    used_p, used_g = set(), set()
    for pi, gi in sorted(np.argwhere(ious >= iou_thr).tolist(), key=lambda ij: -ious[ij[0], ij[1]]):
        if pi in used_p or gi in used_g:
            continue
        used_p.add(pi); used_g.add(gi)
        tp += int(pred_cls[pi] == gt_cls[gi])
        mis += int(pred_cls[pi] != gt_cls[gi])
    counts = dict(tp=tp, misclassified=mis, fn=len(gts) - len(used_g), fp=len(pred_boxes) - len(used_p), n_gt=len(gts))
    outcome = ('misclassified' if mis else 'missed' if counts['fn'] else
               'false_positive' if counts['fp'] else 'true_positive')
    return outcome, counts


def render(result, gts, names, outcome, counts):
    img = result.plot(line_width=3)
    for c, b in gts:
        x1, y1, x2, y2 = b.astype(int)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 0), 2)
        cv2.putText(img, f'GT:{names[c]}', (x1 + 4, y2 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 0), 2)
    color = (0, 160, 0) if outcome == 'true_positive' else (0, 0, 220)
    cv2.rectangle(img, (0, 0), (img.shape[1], 40), color, -1)
    cv2.putText(img, f"{outcome.upper()}  TP={counts['tp']} MIS={counts['misclassified']} FN={counts['fn']} FP={counts['fp']}",
                (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    return img


def analyse(model, images, tag, max_per_bucket=25):
    out_root = SHOWCASE_DIR / tag
    shutil.rmtree(out_root, ignore_errors=True)
    rows, buckets = [], Counter()
    results = model.predict(source=[str(p) for p in images], stream=True, imgsz=IMGSZ, conf=CONF_THRES,
                            iou=NMS_IOU, save=True, save_txt=True, save_conf=True, device=DEVICE,
                            project=str(PREDICT_PROJECT), name=tag, exist_ok=True, verbose=False)
    for img_path, r in zip(images, results):
        h, w = r.orig_shape
        gts = load_gt(img_path, w, h)
        pb, pc, ps = (r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy())
        outcome, counts = classify_outcome(pb, pc, gts)
        if buckets[outcome] < max_per_bucket:
            dst = out_root / outcome / f'{Path(img_path).stem}.jpg'
            dst.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(dst), render(r, gts, model.names, outcome, counts))
        buckets[outcome] += 1
        rows.append(dict(image=Path(img_path).name, outcome=outcome, **counts,
                         pred=';'.join(f'{model.names[c]}:{s:.2f}' for c, s in zip(pc, ps))))
    out_root.mkdir(parents=True, exist_ok=True)
    with (out_root / 'report.csv').open('w', newline='') as f:
        wr = csv.DictWriter(f, fieldnames=list(rows[0]))
        wr.writeheader()
        wr.writerows(rows)
    print(f'[{tag}] {len(images)} images -> {dict(buckets)}  | saved to {out_root}')
    return out_root


def build_overlap_stress_set(n_images=30, canvas=1280, seed=SEED):
    rng = random.Random(seed)
    test_imgs = [p for p in list_images('test') if label_path_for(p).exists()]
    img_dir, lbl_dir = STRESS_DIR / 'images' / 'stress', STRESS_DIR / 'labels' / 'stress'
    for d in (img_dir, lbl_dir):
        shutil.rmtree(d, ignore_errors=True)
        d.mkdir(parents=True)
    cache = {}

    def cutout(p):
        """Crop the first labelled fruit (GT box) and mask it from its background."""
        if p not in cache:
            img = cv2.imread(str(p))
            h, w = img.shape[:2]
            gts = load_gt(p, w, h)
            if not gts:
                cache[p] = (img[:1, :1], np.zeros((1, 1), np.uint8), 0)
                return cache[p]
            cls, b = gts[0]
            x1, y1, x2, y2 = np.clip(b, 0, [w, h, w, h]).astype(int)
            crop = img[y1:y2, x1:x2]
            mask, _ = segment_fruit(crop)
            if mask.sum() < 0.2 * mask.size * 255:    # segmentation unreliable -> use whole box
                mask = np.full(crop.shape[:2], 255, np.uint8)
            cache[p] = (crop, mask, cls)
        return cache[p]

    out = []
    for i in range(n_images):
        bg = np.full((canvas, canvas, 3), 245, np.uint8)
        labels, prev = [], None
        for p in rng.sample(test_imgs, min(len(test_imgs), rng.randint(3, 6))):
            crop, mask, cls = cutout(p)
            if not mask.any():
                continue
            s = rng.uniform(280, 420) / max(crop.shape[:2])
            crop = cv2.resize(crop, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
            mask = cv2.resize(mask, (crop.shape[1], crop.shape[0]), interpolation=cv2.INTER_NEAREST)
            ch, cw = crop.shape[:2]
            if prev is None:
                cx, cy = rng.randint(cw, canvas - cw), rng.randint(ch, canvas - ch)
            else:                                     # place next to the previous fruit -> overlap
                ang, d = rng.uniform(0, 2 * np.pi), rng.uniform(0.45, 0.8) * (cw + prev[2]) / 2
                cx, cy = int(prev[0] + d * np.cos(ang)), int(prev[1] + d * np.sin(ang))
            x1, y1 = int(np.clip(cx - cw // 2, 0, canvas - cw)), int(np.clip(cy - ch // 2, 0, canvas - ch))
            roi = bg[y1:y1 + ch, x1:x1 + cw]
            roi[mask > 0] = crop[mask > 0]            # later fruit occludes earlier ones
            ys, xs = np.nonzero(mask)
            bx1, by1, bx2, by2 = x1 + xs.min(), y1 + ys.min(), x1 + xs.max(), y1 + ys.max()
            labels.append(f'{cls} {(bx1 + bx2) / 2 / canvas:.6f} {(by1 + by2) / 2 / canvas:.6f} '
                          f'{(bx2 - bx1) / canvas:.6f} {(by2 - by1) / canvas:.6f}')
            prev = (x1 + cw // 2, y1 + ch // 2, cw)
        p_out = img_dir / f'overlap_{i:03d}.jpg'
        cv2.imwrite(str(p_out), bg)
        (lbl_dir / f'{p_out.stem}.txt').write_text('\n'.join(labels) + '\n')
        out.append(p_out)
    print(f'✓ Built {len(out)} overlapping-fruit images')
    return out


print('✓ inference helpers defined')

In [ ]:
model = YOLO(str(BEST_PT))

test_dir = analyse(model, list_images('test'), 'test')
stress_dir = analyse(model, build_overlap_stress_set(30), 'stress_overlap')

# ── Showcase: one True Positive + one failure case ─────────────────────────────
tp = sorted((test_dir / 'true_positive').glob('*.jpg'))[:1]
failures = [p for b in ('missed', 'misclassified', 'false_positive')
            for p in sorted((stress_dir / b).glob('*.jpg'))][:1]
if not failures:   # model was perfect on the stress set -> look at test-set errors
    failures = [p for b in ('misclassified', 'missed', 'false_positive') for p in sorted((test_dir / b).glob('*.jpg'))][:1]
show_images(tp + failures,
            ['✅ True positive (test set)'] * len(tp) + [f'❌ Failure: {p.parent.name}' for p in failures],
            cols=2, size=8)

## 11 · Export to ONNX (deployment prep)
Export `best.pt` to ONNX with graph simplification, check it in raw **ONNX Runtime**, and benchmark it against PyTorch. Output: `Drive/exports/yolov8s_dates.onnx`.

In [ ]:
import time
import onnxruntime as ort

DYNAMIC, HALF = False, False      # HALF=True -> FP16 (GPU only); DYNAMIC=True -> variable batch/resolution

exported = Path(YOLO(str(BEST_PT)).export(format='onnx', imgsz=IMGSZ, dynamic=DYNAMIC,
                                          simplify=True, half=HALF, device=DEVICE))
ONNX_PATH = EXPORT_DIR / f'{RUN_NAME}.onnx'
shutil.copy2(exported, ONNX_PATH)

# 1) Verify in raw ONNX Runtime (what a production service would use)
providers = [p for p in ('CUDAExecutionProvider', 'CPUExecutionProvider') if p in ort.get_available_providers()]
sess = ort.InferenceSession(str(ONNX_PATH), providers=providers)
inp = sess.get_inputs()[0]
dummy = np.random.rand(1, 3, IMGSZ, IMGSZ).astype(np.float16 if 'float16' in inp.type else np.float32)
out = sess.run(None, {inp.name: dummy})
print(f'✓ ORT OK: {inp.name}{inp.shape} -> {out[0].shape} via {sess.get_providers()[0]}')

# 2) End-to-end latency (pre-process + inference + NMS)
sample_img = str(list_images('test')[0])
bench = {}
for tag, path in (('pt', BEST_PT), ('onnx', ONNX_PATH)):
    m = YOLO(str(path), task='detect')
    for _ in range(5):
        m.predict(sample_img, imgsz=IMGSZ, device=DEVICE, verbose=False)
    t0 = time.perf_counter()
    for _ in range(50):
        m.predict(sample_img, imgsz=IMGSZ, device=DEVICE, verbose=False)
    bench[f'latency_ms_{tag}'] = round((time.perf_counter() - t0) / 50 * 1000, 2)

info = dict(onnx=str(ONNX_PATH), size_mb=round(ONNX_PATH.stat().st_size / 1e6, 2), imgsz=IMGSZ,
            dynamic=DYNAMIC, half=HALF, provider=sess.get_providers()[0], names=CLASS_NAMES, **bench)
(EXPORT_DIR / 'export_info.json').write_text(json.dumps(info, indent=2))
print(json.dumps({k: v for k, v in info.items() if k != 'names'}, indent=2))

## 12 · Try your own image (optional)

In [ ]:
from google.colab import files

uploaded = files.upload()                       # pick one or more .jpg/.png files
infer_model = YOLO(str(EXPORT_DIR / f'{RUN_NAME}.onnx'), task='detect')   # or YOLO(str(BEST_PT))
saved = []
for fname in uploaded:
    r = infer_model.predict(fname, imgsz=IMGSZ, conf=CONF_THRES, iou=NMS_IOU, save=True,
                            project=str(PREDICT_PROJECT), name='uploads', exist_ok=True, verbose=False)[0]
    print(fname, '->', [(infer_model.names[int(c)], round(float(s), 2)) for c, s in zip(r.boxes.cls, r.boxes.conf)])
    saved.append(Path(r.save_dir) / Path(fname).name)
show_images(saved, cols=2, size=7)

## ✅ Done: what's on your Google Drive

```
MyDrive/Date_Vision_Project/
├── data/qa/                            # label_qa_grid.jpg, the training .yaml, needs_review.csv, manifest.csv
├── weights/yolov8s.pt                  # pretrained COCO weights
├── runs/train/yolov8s_dates/           # best.pt, last.pt, results.png, curves
├── runs/eval/yolov8s_dates_test/       # confusion matrices, PR/F1 curves, metrics.json
├── runs/predict/                       # raw Ultralytics predictions
├── showcase/{test,stress_overlap}/     # TP / failure images + report.csv
└── exports/                            # yolov8s_dates.onnx, export_info.json
```